# CoAD on Kaggle: M6-B (part 24 of CoAD_experiments.ipynb)

Run all cells (or Save Version > Save & Run All). Same code as the Colab notebook; only the first cell differs (data from a Kaggle dataset). Download `to_drive.zip` from the Output tab at the end.

In [ ]:
import json, re, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

# Kaggle instead of Google Drive: the Drive folder CoAD_project is attached as a private dataset (read-only).
# Small folders are copied to a writable CoAD_project; the data archive and the saved models are read in place.
# Result files of earlier versions of this notebook, attached as inputs, are added back so finished runs are skipped.
KAGGLE_TAG = 'kaggle-m6b'
INPUT = Path('/kaggle/input')
candidates = [p.parents[1] for p in INPUT.rglob('env/commit.txt')
              if (p.parents[1] / 'data_source' / 'TSB-AD-U.zip').is_file()]
assert candidates, 'Attach the dataset with the CoAD_project folder (env/, eda/, results/, data_source/)'
source = max(candidates, key=lambda d: (d / 'runs' / 'reimpl').is_dir())
STORE = Path('/kaggle/working/CoAD_project')
if not STORE.exists():
    for sub in ['env', 'eda', 'results']:
        shutil.copytree(source / sub, STORE / sub)
    (STORE / 'data_source').symlink_to(source / 'data_source', target_is_directory=True)
    (STORE / 'runs').mkdir()
    if (source / 'runs' / 'reimpl').is_dir():
        (STORE / 'runs' / 'reimpl').symlink_to(source / 'runs' / 'reimpl', target_is_directory=True)
    latest = {}
    for csv in INPUT.rglob('results/*.csv'):
        if csv.name not in latest or csv.stat().st_size > latest[csv.name].stat().st_size:
            latest[csv.name] = csv
    for name, csv in latest.items():
        shutil.copy2(csv, STORE / 'results' / name)
print('Dataset:', source, '| result files:', len(list((STORE / 'results').glob('*.csv'))),
      '| saved models:', (STORE / 'runs' / 'reimpl').exists())
REPO = Path('/tmp/CoAD')
commit = (STORE / 'env' / 'commit.txt').read_text(encoding='utf-8').strip()

if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/richard-tang199/CoAD.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '-q', commit], cwd=REPO, check=True)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())

In [ ]:
# Packages imported by baseline_compare.py, the evaluator and CoAD; checked in a fresh process
CHECK_IMPORTS = 'import arch, stumpy, tslearn, sktime, torch_geometric, statsmodels, numba'

def deps_ok():
    return subprocess.run([sys.executable, '-c', CHECK_IMPORTS], capture_output=True).returncode == 0

if not deps_ok():
    install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')],
                             capture_output=True, text=True)
    (STORE / 'env' / 'install_experiments.log').write_text(install.stdout + install.stderr, encoding='utf-8')
    print('pip exit code:', install.returncode)
    assert install.returncode == 0, 'Installation failed; inspect env/install_experiments.log'
assert deps_ok(), 'Some packages still cannot be imported; inspect env/install_experiments.log'

# Use the current tslearn (about 3x faster than 0.6.3); SAND's short last batch is handled in 23.3
tslearn_version = subprocess.check_output([sys.executable, '-c', 'import tslearn; print(tslearn.__version__)'], text=True).strip()
if tslearn_version.startswith('0.6.'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-deps', 'tslearn'], check=True)
    tslearn_version = subprocess.check_output([sys.executable, '-c', 'import tslearn; print(tslearn.__version__)'], text=True).strip()
print('All required packages import in this runtime | tslearn', tslearn_version)

In [ ]:
import zipfile
archive = STORE / 'data_source' / 'TSB-AD-U.zip'
RAW = REPO / 'dataset' / 'TSB-AD' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for m in z.namelist():
        if m.startswith('TSB-AD-U/') and m.endswith('.csv') and '..' not in m:
            target = RAW / Path(m).name
            if not target.exists():
                with z.open(m) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
RAW_FILES = sorted(p.name for p in RAW.iterdir())
assert len(RAW_FILES) == 870, f'raw/ must contain exactly 870 CSVs, found {len(RAW_FILES)}'

baseline_script = REPO / 'baseline_compare.py'
required_imports = 'import os\nimport numpy as np\nimport torch\n'
script_source = baseline_script.read_text(encoding='utf-8')
if not script_source.startswith(required_imports):
    baseline_script.write_text(required_imports + script_source, encoding='utf-8')

# NumPy 2.0 removed the aliases np.Inf / np.NaN; the baselines use them (same values as np.inf / np.nan)
NUMPY2_RENAMES = {'np.Inf': 'np.inf', 'np.NaN': 'np.nan'}
for rel in ['baseline/sand.py', 'baseline/feature.py']:
    path = REPO / rel
    source = path.read_text(encoding='utf-8')
    for old, new in NUMPY2_RENAMES.items():
        source = source.replace(old, new)
    path.write_text(source, encoding='utf-8')
# SAND clusters each online batch into k = 6 groups; the last batch of some series has fewer than 6
# subsequences and tslearn >= 0.8 then fails. Use at most as many clusters as subsequences in that batch.
sand_file = REPO / 'baseline' / 'sand.py'
sand_source = sand_file.read_text(encoding='utf-8')
old_line = 'ks = KShape(n_clusters=self.k, verbose=False)'
new_line = 'ks = KShape(n_clusters=min(self.k, len(all_subsequences)), verbose=False)'
if old_line in sand_source:
    sand_file.write_text(sand_source.replace(old_line, new_line), encoding='utf-8')
assert new_line in sand_file.read_text(encoding='utf-8'), 'SAND patch not applied'

diff_text = subprocess.check_output(['git', 'diff', '--', 'baseline_compare.py', 'baseline/sand.py', 'baseline/feature.py'],
                                    cwd=REPO, text=True)
(STORE / 'env' / 'baseline_compat.diff').write_text(diff_text, encoding='utf-8')
changed = sum(1 for line in diff_text.splitlines() if line.startswith('+') and not line.startswith('+++'))
print('Data ready:', len(RAW_FILES), 'files |', changed, 'lines added/changed in baseline code (env/baseline_compat.diff)')

In [ ]:
RESULTS_DIR = STORE / 'results'   # each session appends to its own file, set in 23.6
RUNS = STORE / 'runs' / 'tsbad64'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
METRIC_KEYS = ['f1_score', 'auc_prc', 'auc_roc', 'R_AUC_ROC', 'R_AUC_PR', 'VUS_ROC', 'VUS_PR']
COLUMNS = ['method', 'group', 'status', 'seconds', 'run_id', 'commit', 'error'] + METRIC_KEYS

def read_all_results():
    """All result files of all sessions, oldest run first (run_id starts with the UTC start time)."""
    files = sorted(RESULTS_DIR.glob('tsbad64_results*.csv'))
    if not files:
        return pd.DataFrame(columns=COLUMNS)
    frames = [pd.read_csv(f).assign(source_file=f.name) for f in files]
    return pd.concat(frames, ignore_index=True).sort_values('run_id', kind='stable').reset_index(drop=True)
KEEP_MODELS = False   # True also copies CoAD .pth files (~MBs per series) to Drive

def summarize_metrics(metric_file):
    m = json.loads(Path(metric_file).read_text())
    row = {k: m['point_wise'][k] for k in ['f1_score', 'auc_prc', 'auc_roc']}
    row.update(m['vus'])
    return row

def command_for(method, group, timestamp):
    if method == 'CoAD':   # flags from runners/run_all_CoAD.sh (TSB-AD lines)
        return [sys.executable, 'train_classify_mask.py', '--dataset_name', 'TSB-AD', '--group_name', str(group),
                '--model_name', 'freq_tf_v2', '--timestamp', timestamp, '--device', 'cuda:0', '--plot', 'False',
                '--num_epochs', '300', '--batch_size', '128', '--window_multiple', '4', '--lr', '2e-3',
                '--patch_size', '8', '--omega', '10', '--random_seed', '46']
    return [sys.executable, 'baseline_compare.py', '--model_name', method,   # runners/run_dm_baseline.sh
            '--dataset_name', 'TSB-AD', '--group_name', str(group)]

def run_one(method, group):
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_id = f'{timestamp}_{method}_TSB-AD_{group}'
    out = RUNS / run_id
    out.mkdir(parents=True)
    cmd = command_for(method, group, timestamp)
    (out / 'command.json').write_text(json.dumps(cmd), encoding='utf-8')
    started = time.perf_counter()
    with (out / 'console.log').open('w', encoding='utf-8') as log:
        result = subprocess.run(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    row = {'method': method, 'group': group, 'run_id': run_id, 'commit': commit,
           'seconds': round(time.perf_counter() - started, 1)}
    try:
        assert result.returncode == 0, f'exit code {result.returncode}'
        g3 = str(group).zfill(3)
        if method == 'CoAD':
            src = REPO / 'mask_results' / 'TSB-AD' / f'freq_tf_v2_max_{timestamp}'
            ignore = None if KEEP_MODELS else shutil.ignore_patterns('*.pth')
            shutil.copytree(src, out / 'artifacts', ignore=ignore)
            shutil.rmtree(src)   # free local disk
            metric_file = next((out / 'artifacts' / 'all' / 'data').glob('group_*.json'))
        else:
            src = REPO / 'baseline_result' / 'TSB-AD' / method
            for rel in [f'data/group_{g3}.json', f'anomaly_score/group_{g3}.npy']:
                (out / 'artifacts' / rel).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src / rel, out / 'artifacts' / rel)
            metric_file = out / 'artifacts' / f'data/group_{g3}.json'
        row.update(summarize_metrics(metric_file), status='success', error='')
    except Exception as exc:
        lines = (out / 'console.log').read_text(encoding='utf-8', errors='replace').strip().splitlines()
        row.update(status='failed', error=f'{exc} | ' + ' / '.join(lines[-3:]))
    pd.DataFrame([row]).reindex(columns=COLUMNS).to_csv(
        RESULTS, mode='a', header=not RESULTS.exists(), index=False)
    return row

In [ ]:
def tsbad_groups(runner_name):
    text = (REPO / 'runners' / runner_name).read_text(encoding='utf-8')
    groups = []
    for block in text.split('for g in $(seq ')[1:]:
        start, end = map(int, re.match(r'(\d+) (\d+)\)', block).groups())
        if '--dataset_name TSB-AD' in block.split('done')[0]:
            groups += list(range(start, end + 1))
    return groups

GROUPS = tsbad_groups('run_all_CoAD.sh')
assert GROUPS == tsbad_groups('run_dm_baseline.sh') and len(GROUPS) == 64, 'Runners disagree on the TSB-AD groups'

manifest_file = STORE / 'eda' / 'manifest_tsbad64.csv'
if not manifest_file.exists():
    rows = []
    for g in GROUPS:
        name = RAW_FILES[g - 1]
        n_points = len(pd.read_csv(RAW / name).dropna())
        rows.append({'dataset': 'TSB-AD', 'group': g, 'source_file': name, 'family': name.split('_')[1],
                     'n_train': int(name.split('.')[0].split('_')[-3]), 'n_test': n_points,
                     'selection_reason': 'Full TSB-AD selection of Table 1 (runner loops)', 'commit': commit})
    pd.DataFrame(rows).to_csv(manifest_file, index=False)
MANIFEST = pd.read_csv(manifest_file)
display(MANIFEST.groupby('family').agg(series=('group', 'count'), test_points=('n_test', 'sum')))

In [ ]:
ABLATION_VARIANTS = {   # model_name -> (Table 2 row, extra flags of runners/run_ablation_CoAD.sh)
    'time_soft':        ('OE (Time) + MAE (Guide w/ Soft Mask)', []),
    'guide_hard':       ('OE+MAE (Guide w/ Hard Mask)', ['--window_multiple', '4']),
    'freq_time_mean':   ('Decision_Mean', []),
    'freq_hard':        ('OE+MAE (Random)', []),
    'freq_gating':      ('OE+MAE (Grating)', []),
    'freq_time_add':    ('Feature_Add', []),
    'freq_time_gate':   ('Feature_Gate', []),
    'freq_time_point':  ('Step Level', ['--window_multiple', '4']),
    'freq_time_window': ('Window Level', ['--window_multiple', '1']),
}
# Table 2, TSB-AD columns (%). Rows of the 'Cooperative' block matched by row order (check against the PDF).
PAPER_T2 = pd.DataFrame({
    'CoAD':             [49.13, 43.66, 40.50, 39.83],
    'time_soft':        [43.10, 36.87, 37.91, 36.92],
    'guide_hard':       [44.06, 38.57, 35.31, 34.76],
    'freq_time_mean':   [42.24, 36.23, 33.71, 33.18],
    'freq_hard':        [45.01, 39.31, 35.79, 35.20],
    'freq_gating':      [47.07, 41.48, 37.39, 37.08],
    'freq_time_add':    [40.11, 35.15, 33.40, 32.94],
    'freq_time_gate':   [42.23, 37.20, 34.87, 34.40],
    'freq_time_point':  [42.52, 36.06, 34.02, 33.37],
    'freq_time_window': [22.43, 17.08, 15.71, 15.53],
}, index=['F1', 'AUC-PR', 'R-AUC-PR', 'VUS-PR']).T

ABL_RUNS = STORE / 'runs' / 'ablation'
ABL_RUNS.mkdir(parents=True, exist_ok=True)
ABL_COLUMNS = ['variant', 'group', 'status', 'seconds', 'run_id', 'commit', 'error'] + METRIC_KEYS

def read_ablation():
    files = sorted(RESULTS_DIR.glob('ablation_results*.csv'))
    if not files:
        return pd.DataFrame(columns=ABL_COLUMNS)
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True).sort_values('run_id', kind='stable')

def run_variant(variant, group, results_file):
    """Same command as the CoAD runs of 23.4; only --model_name (and the runner's extra flags) change."""
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_id = f'{timestamp}_{variant}_TSB-AD_{group}'
    out = ABL_RUNS / run_id
    out.mkdir(parents=True)
    cmd = command_for('CoAD', group, timestamp)
    cmd[cmd.index('--model_name') + 1] = variant
    cmd += ABLATION_VARIANTS[variant][1]   # argparse keeps the last value of a repeated flag
    (out / 'command.json').write_text(json.dumps(cmd), encoding='utf-8')
    started = time.perf_counter()
    with (out / 'console.log').open('w', encoding='utf-8') as log:
        result = subprocess.run(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    row = {'variant': variant, 'group': group, 'run_id': run_id, 'commit': commit,
           'seconds': round(time.perf_counter() - started, 1)}
    try:
        assert result.returncode == 0, f'exit code {result.returncode}'
        src = REPO / 'mask_results' / 'TSB-AD' / f'{variant}_max_{timestamp}'
        shutil.copytree(src, out / 'artifacts', ignore=None if KEEP_MODELS else shutil.ignore_patterns('*.pth'))
        shutil.rmtree(src)
        metric_file = next((out / 'artifacts' / 'all' / 'data').glob('group_*.json'))
        row.update(summarize_metrics(metric_file), status='success', error='')
    except Exception as exc:
        lines = (out / 'console.log').read_text(encoding='utf-8', errors='replace').strip().splitlines()
        row.update(status='failed', error=f'{exc} | ' + ' / '.join(lines[-3:]))
    pd.DataFrame([row]).reindex(columns=ABL_COLUMNS).to_csv(
        results_file, mode='a', header=not results_file.exists(), index=False)
    return row

In [ ]:
ABL_TAG = KAGGLE_TAG                 # different in each notebook copy running at the same time
RUN_VARIANTS = ['guide_hard', 'freq_time_mean']      # one variant per copy: 'time_soft' / 'guide_hard' / 'freq_time_mean'
RUN_GROUPS_ABL = GROUPS           # all 64; MANIFEST.query("family == 'TODS'").group.tolist() for a quick first test
TIME_BUDGET_MIN = 600   # Kaggle: stop well before the 12 h limit
ABL_RESULTS = RESULTS_DIR / f'ablation_results_{ABL_TAG}.csv'

done = read_ablation().query("status == 'success'")
done = set(zip(done['variant'], done['group'].astype(int)))
todo = [(v, g) for v in RUN_VARIANTS for g in RUN_GROUPS_ABL if (v, g) not in done]
print(f'{len(todo)} runs to do | writes to {ABL_RESULTS.name}')
session_start = time.perf_counter()
for i, (variant, group) in enumerate(todo, 1):
    if (time.perf_counter() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue.')
        break
    row = run_variant(variant, group, ABL_RESULTS)
    print(f"[{i}/{len(todo)}] {variant} group {group}: {row['status']} {row['seconds']:.0f}s "
          f"VUS-PR={row.get('VUS_PR', float('nan')):.4f}", flush=True)

In [ ]:
NAMES = {'f1_score': 'F1', 'auc_prc': 'AUC-PR', 'R_AUC_PR': 'R-AUC-PR', 'VUS_PR': 'VUS-PR'}
abl = read_ablation().query("status == 'success'").drop_duplicates(['variant', 'group'], keep='last')
coad = (read_all_results().query("method == 'CoAD' and status == 'success'")
        .drop_duplicates('group', keep='last').set_index('group'))
rows = []
for variant, runs in abl.groupby('variant'):
    groups = runs['group'].astype(int).tolist()
    ours = 100 * runs.set_index('group')[list(NAMES)].mean()
    base = 100 * coad.loc[groups, list(NAMES)].mean()
    for key, name in NAMES.items():
        rows.append({'variant': variant, 'Table 2 row': ABLATION_VARIANTS[variant][0], 'series': len(groups),
                     'metric': name, 'ours': ours[key], 'ours - CoAD': ours[key] - base[key],
                     'paper': PAPER_T2.loc[variant, name],
                     'paper - CoAD': PAPER_T2.loc[variant, name] - PAPER_T2.loc['CoAD', name]})
m6b = pd.DataFrame(rows)
m6b['same direction'] = np.sign(m6b['ours - CoAD']) == np.sign(m6b['paper - CoAD'])
(STORE / 'reports').mkdir(exist_ok=True)
m6b.to_csv(STORE / 'reports' / 'm6b_ablation.csv', index=False)
display(m6b.round(2))

In [ ]:
# Files to put back into the Drive folder CoAD_project (same sub-folders): download to_drive.zip from the Output tab.
# Only this notebook's own result files (tag kaggle-m6b), so nothing written on Colab meanwhile is overwritten.
OUT = Path('/kaggle/working/to_drive')
shutil.rmtree(OUT, ignore_errors=True)
(OUT / 'results').mkdir(parents=True)
for csv in (STORE / 'results').glob(f'*{KAGGLE_TAG}*.csv'):
    shutil.copy2(csv, OUT / 'results' / csv.name)
for sub in ['reports'] + ['runs/ablation']:
    if (STORE / sub).exists():
        shutil.copytree(STORE / sub, OUT / sub)
shutil.make_archive(str(OUT), 'zip', OUT)
print('to_drive.zip:', sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*.csv')))